In [1]:
import numpy as np
from itertools import product
import pandas as pd

In [4]:
dataset = pd.read_csv("/Users/thangmai/Downloads/UNIMELB/HONOURS RESEARCH (ACTL40001)/OldCodebase/dataset/freMTPL2freq.csv")
area_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6
}

dataset["Area"] = dataset["Area"].map(area_mapping)
dataset["VehPower"] = dataset["VehPower"].clip(upper = 9)
dataset["DrivAge"] = pd.cut(
    dataset["DrivAge"],
    bins=[18, 21, 26, 31, 41, 51, 71, np.inf],
    labels=range(7),          # 0 -> 6
    right=False,
).astype(int)
dataset["VehAge"] = pd.cut(
    dataset["VehAge"],
    bins=[0, 1, 11, np.inf],
    labels=range(3),          # 0 -> 2
    right=False,
).astype(int)
dataset["BonusMalus"] = dataset["BonusMalus"].clip(upper=150)
dataset["log_Density"] = np.log(dataset["Density"])
dataset = dataset.drop("Density", axis= 1)
# Binary: single 0/1 column
dataset["VehGas"] = (dataset["VehGas"] == "Diesel").astype(int)

# Multi-level categoricals: one-hot encode
cat_cols = ["VehPower", "VehAge", "DrivAge", "VehBrand", "Region"]
dataset = pd.get_dummies(dataset, columns=cat_cols, prefix=cat_cols, dtype=int, drop_first= True)

y = dataset["ClaimNb"].to_numpy()
exposure = dataset["Exposure"].to_numpy()
X = dataset.drop(columns=["IDpol", "ClaimNb", "Exposure"])   # 48 columns

assert X.shape[1] == 48

In [70]:
dataset.to_csv("freMTPL2freq_model_ready.csv")

In [ ]:
sev_dataset = pd.read_csv("/Users/thangmai/Downloads/UNIMELB/HONOURS RESEARCH (ACTL40001)/OldCodebase/dataset/freMTPL2sev.csv")
sev_dataset = sev_dataset.merge(dataset, on = "IDpol")
sev_dataset = sev_dataset.drop("ClaimNb", axis = 1)

In [85]:
sev_dataset.to_csv("freMTPL2sev_model_ready.csv")